# Module 3: Production Deployment with Amazon Bedrock AgentCore

## Overview

In this module, you'll deploy the **Product Catalog Agent** from Module 01 to production using **Amazon Bedrock AgentCore**. The local prototype's mock identity (`UserSession`) maps to real JWT-based authentication, and the MCP server becomes a Lambda function behind AgentCore Gateway.

### What You'll Build

```
                     ┌──────────────────────────────────────────────┐
                     │           Cognito User Pool                  │
                     │  ┌────────────┐    ┌────────────┐           │
                     │  │  customer   │    │   admin    │           │
                     │  │   group     │    │   group    │           │
                     │  └────────────┘    └────────────┘           │
                     └───────────────┬──────────────────────────────┘
                                     │ JWT (cognito:groups)
                                     ▼
┌──────────┐  invoke  ┌──────────────────────────────────┐
│ Streamlit │ ──────► │     AgentCore Runtime             │
│   App     │         │  ┌────────────────────────────┐   │
└──────────┘          │  │  Product Catalog Agent      │   │
                      │  │  • JWT role extraction       │   │
                      │  │  • Tool filtering by role    │   │
                      │  │  • Role-aware system prompt  │   │
                      │  └──────────┬─────────────────┘   │
                      └─────────────┼─────────────────────┘
                                    │ SigV4 + Bearer token
                                    ▼
                      ┌─────────────────────────────────────────┐
                      │        AgentCore Gateway                 │
                      │   ┌─────────────────────────────────┐   │
                      │   │  RBAC Interceptor Lambda         │   │
                      │   │  • REQUEST: blocks admin tools   │   │
                      │   │    for customer role              │   │
                      │   │  • RESPONSE: filters tools/list  │   │
                      │   │    to hide admin tools            │   │
                      │   └─────────────────────────────────┘   │
                      │                                         │
                      │   ┌─────────────────────────────────┐   │
                      │   │  ProductTools Target (Lambda)    │   │
                      │   │  11 MCP tools via Lambda         │   │
                      │   └───────────────┬─────────────────┘   │
                      └───────────────────┼─────────────────────┘
                                          │
                                          ▼
                      ┌───────────────────────────────────┐
                      │      DynamoDB Products Table      │
                      └───────────────────────────────────┘
```

### Prototype to Production Mapping

| Module 01 (Prototype) | Module 03 (Production) |
|----------------------|------------------------|
| `UserSession` class | Cognito JWT token |
| `role` field | `cognito:groups` claim |
| Local MCP server (stdio) | Lambda behind AgentCore Gateway |
| Local tool filtering | **Defense-in-depth**: Agent filtering + Gateway interceptor |
| `python agent.py` | AgentCore Runtime (container) |

### Defense-in-Depth RBAC

Two independent layers enforce access control:
1. **Agent-side** (same as Module 01): Tool filtering removes admin tools from customer agents
2. **Gateway interceptor** (new): Lambda intercepts requests/responses at infrastructure level

### Learning Objectives
1. Deploy Lambda functions as MCP tool backends via AgentCore Gateway
2. Configure Cognito User Pool with groups for JWT-based RBAC
3. Implement a Gateway interceptor for infrastructure-level access control
4. Build and deploy a containerized agent to AgentCore Runtime
5. Test role-based access control end-to-end in production

### Prerequisites
- Module 01 completed (DynamoDB products table populated)
- AWS credentials with AgentCore, Lambda, Cognito, ECR, and IAM permissions
- Docker installed (for building container images)

### Time: ~40 minutes

## Step 1: Setup and Configuration

In [ ]:
# Install dependencies
!pip install -q boto3 strands-agents strands-agents-tools mcp requests

In [ ]:
import boto3
import json
import os
import time
import uuid

# Load region from previous module or use default
try:
    %store -r REGION
    print(f"Loaded REGION from previous module: {REGION}")
except:
    session = boto3.Session()
    REGION = session.region_name or 'us-west-2'
    print(f"Using default region: {REGION}")

os.environ['AWS_REGION'] = REGION
os.environ['AWS_DEFAULT_REGION'] = REGION

# Workshop configuration
WORKSHOP_PREFIX = 'ecommerce-workshop'
PRODUCTS_TABLE = f'{WORKSHOP_PREFIX}-products'

# Naming conventions
LAMBDA_ROLE_NAME = f'{WORKSHOP_PREFIX}-lambda-role'
GATEWAY_ROLE_NAME = f'{WORKSHOP_PREFIX}-gateway-role'
GATEWAY_NAME = f'{WORKSHOP_PREFIX}-product-gateway'
RUNTIME_ROLE_NAME = f'{WORKSHOP_PREFIX}-runtime-role'
RUNTIME_NAME = 'ecommerce_workshop_product_catalog_agent'  # underscores only - AgentCore Runtime name constraint
COGNITO_POOL_NAME = f'{WORKSHOP_PREFIX}-user-pool'
ECR_REPO_NAME = f'{WORKSHOP_PREFIX}-product-catalog-agent'
TEST_PASSWORD = 'Workshop1234!'

print(f"\nConfiguration:")
print(f"  Region: {REGION}")
print(f"  Products Table: {PRODUCTS_TABLE}")
print(f"  Gateway: {GATEWAY_NAME}")

In [ ]:
# Verify AWS credentials
sts = boto3.client('sts', region_name=REGION)
identity = sts.get_caller_identity()
ACCOUNT_ID = identity['Account']

print(f"AWS Account: {ACCOUNT_ID}")
print(f"AWS Identity: {identity['Arn']}")

## Step 2: Create IAM Roles

We need three IAM roles:
1. **Lambda execution role** - Allows Lambda functions to access DynamoDB
2. **Gateway role** - Allows AgentCore Gateway to invoke Lambda functions
3. **Runtime role** - Allows AgentCore Runtime to invoke Bedrock models and connect to Gateway

In [ ]:
from utils import create_lambda_execution_role

iam_client = boto3.client('iam', region_name=REGION)

# DynamoDB table ARN
products_table_arn = f'arn:aws:dynamodb:{REGION}:{ACCOUNT_ID}:table/{PRODUCTS_TABLE}'

# Create Lambda execution role
print("Creating Lambda execution role...")
lambda_role_resp = create_lambda_execution_role(
    iam_client,
    LAMBDA_ROLE_NAME,
    [products_table_arn]
)
LAMBDA_ROLE_ARN = lambda_role_resp['Role']['Arn']
print(f"Lambda Role ARN: {LAMBDA_ROLE_ARN}")

## Step 3: Deploy Lambda Functions

We deploy two Lambda functions:
1. **Product Tools Lambda** - Contains all 11 MCP tools (same tools as Module 01's MCP server)
2. **RBAC Interceptor Lambda** - Gateway interceptor that enforces role-based access control

### Product Tools Lambda
This Lambda receives tool calls from AgentCore Gateway and routes them to the appropriate handler. The tool name comes from `context.client_context.custom['bedrockAgentCoreToolName']`.

### RBAC Interceptor Lambda
This Lambda runs at two interception points on the Gateway:
- **REQUEST**: Blocks `tools/call` for admin tools when the caller is a customer
- **RESPONSE**: Filters `tools/list` to hide admin tools from customer sessions

In [ ]:
from utils import create_lambda_function

lambda_client = boto3.client('lambda', region_name=REGION)

# Environment variables for product tools Lambda
lambda_env_vars = {
    'PRODUCTS_TABLE_NAME': PRODUCTS_TABLE
}

# Deploy Product Tools Lambda (11 tools)
print("Deploying Product Tools Lambda...")
product_tools_result = create_lambda_function(
    lambda_client,
    f'{WORKSHOP_PREFIX}-product-tools',
    LAMBDA_ROLE_ARN,
    'lambda_tools/product_tools_lambda.py',
    'product_tools_lambda.lambda_handler',
    lambda_env_vars,
    REGION
)
PRODUCT_TOOLS_ARN = product_tools_result['function_arn']
print(f"Product Tools ARN: {PRODUCT_TOOLS_ARN}\n")

# Deploy RBAC Interceptor Lambda
print("Deploying RBAC Interceptor Lambda...")
interceptor_result = create_lambda_function(
    lambda_client,
    f'{WORKSHOP_PREFIX}-rbac-interceptor',
    LAMBDA_ROLE_ARN,
    'lambda_tools/rbac_interceptor_lambda.py',
    'rbac_interceptor_lambda.lambda_handler',
    {},  # No env vars needed - interceptor is stateless
    REGION
)
INTERCEPTOR_ARN = interceptor_result['function_arn']
print(f"RBAC Interceptor ARN: {INTERCEPTOR_ARN}")

## Step 4: Set Up Cognito User Pool with Groups

In Module 01, we used a `UserSession` class with a `role` field. In production, user roles come from **Cognito groups** embedded in the JWT token's `cognito:groups` claim.

We create:
- A Cognito User Pool with `customer` and `admin` groups
- An app client for user login (USER_PASSWORD_AUTH for the Streamlit app)
- An app client for M2M authentication (client_credentials for Gateway access)
- Test users: one customer, one admin

In [ ]:
from utils import (
    get_or_create_cognito_user_pool,
    create_cognito_group,
    get_or_create_cognito_resource_server,
    get_or_create_cognito_app_client,
    get_or_create_user_app_client,
    get_or_create_cognito_domain,
    create_test_user
)

cognito_client = boto3.client('cognito-idp', region_name=REGION)

# 1. Create User Pool
print("Setting up Cognito User Pool...")
USER_POOL_ID = get_or_create_cognito_user_pool(cognito_client, COGNITO_POOL_NAME)

# 2. Create groups for RBAC
print("\nCreating RBAC groups...")
create_cognito_group(cognito_client, USER_POOL_ID, 'customer', 'Read-only product catalog access')
create_cognito_group(cognito_client, USER_POOL_ID, 'admin', 'Full product catalog access (read + write)')

# 3. Create resource server and M2M app client (for Gateway JWT validation)
COGNITO_RESOURCE_SERVER_ID = f'{WORKSHOP_PREFIX}-gateway-api'
SCOPES = [
    {"ScopeName": "gateway:read", "ScopeDescription": "Read access to Gateway tools"},
    {"ScopeName": "gateway:write", "ScopeDescription": "Write access to Gateway tools"}
]
get_or_create_cognito_resource_server(
    cognito_client, USER_POOL_ID, COGNITO_RESOURCE_SERVER_ID,
    f"{WORKSHOP_PREFIX} Gateway API", SCOPES
)

M2M_CLIENT_NAME = f'{WORKSHOP_PREFIX}-gateway-client'
M2M_CLIENT_ID, M2M_CLIENT_SECRET = get_or_create_cognito_app_client(
    cognito_client, USER_POOL_ID, M2M_CLIENT_NAME, COGNITO_RESOURCE_SERVER_ID, SCOPES
)

# 4. Create user-facing app client (for Streamlit login)
USER_CLIENT_NAME = f'{WORKSHOP_PREFIX}-user-client'
USER_CLIENT_ID = get_or_create_user_app_client(
    cognito_client, USER_POOL_ID, USER_CLIENT_NAME
)

# 5. Create Cognito domain for OAuth
DOMAIN_PREFIX = f"{WORKSHOP_PREFIX}-{str(uuid.uuid4())[:8]}"
get_or_create_cognito_domain(cognito_client, USER_POOL_ID, DOMAIN_PREFIX)

# 6. Create test users with group membership
print("\nCreating test users...")
CUSTOMER_EMAIL = 'john.customer@example.com'
ADMIN_EMAIL = 'alice.admin@example.com'

create_test_user(
    cognito_client, USER_POOL_ID, CUSTOMER_EMAIL, TEST_PASSWORD,
    group_name='customer', name='John Smith'
)
create_test_user(
    cognito_client, USER_POOL_ID, ADMIN_EMAIL, TEST_PASSWORD,
    group_name='admin', name='Alice Admin'
)

# Discovery URL for JWT validation
COGNITO_DISCOVERY_URL = f'https://cognito-idp.{REGION}.amazonaws.com/{USER_POOL_ID}/.well-known/openid-configuration'

print(f"\nCognito Configuration:")
print(f"  User Pool ID: {USER_POOL_ID}")
print(f"  Groups: customer, admin")
print(f"  M2M Client ID: {M2M_CLIENT_ID}")
print(f"  User Client ID: {USER_CLIENT_ID}")
print(f"  Test Customer: {CUSTOMER_EMAIL}")
print(f"  Test Admin: {ADMIN_EMAIL}")

## Step 5: Create AgentCore Gateway with RBAC Interceptor

The AgentCore Gateway exposes our Lambda tools as MCP endpoints. We configure it with:
- **JWT authorizer** using Cognito for authentication
- **RBAC interceptor** Lambda for access control enforcement

The interceptor runs at two points:
1. **REQUEST**: Before a tool is called - blocks admin tools for customer role
2. **RESPONSE**: After tools/list - filters out admin tools for customer role

In [ ]:
from utils import create_agentcore_gateway_role, create_gateway

gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)

# Create Gateway IAM role (needs permission to invoke both Lambda functions)
print("Creating Gateway IAM role...")
gateway_role_resp = create_agentcore_gateway_role(
    iam_client,
    GATEWAY_ROLE_NAME,
    [PRODUCT_TOOLS_ARN, INTERCEPTOR_ARN]
)
GATEWAY_ROLE_ARN = gateway_role_resp['Role']['Arn']
print(f"Gateway Role ARN: {GATEWAY_ROLE_ARN}")

In [ ]:
# Create Gateway with JWT auth and RBAC interceptor
# allowedClients validates client_id claim in JWT access tokens
# Both M2M and user app clients are allowed (user access tokens have client_id = USER_CLIENT_ID)
auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [M2M_CLIENT_ID, USER_CLIENT_ID],
        "discoveryUrl": COGNITO_DISCOVERY_URL
    }
}

print("Creating AgentCore Gateway with RBAC interceptor...")
gateway_response = create_gateway(
    gateway_client,
    GATEWAY_NAME,
    GATEWAY_ROLE_ARN,
    auth_config,
    'Product Catalog Gateway with RBAC interceptor',
    interceptor_lambda_arn=INTERCEPTOR_ARN  # Attach RBAC interceptor
)

if gateway_response:
    GATEWAY_ID = gateway_response['gatewayId']
    GATEWAY_URL = gateway_response['gatewayUrl']
    GATEWAY_ARN = f'arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:gateway/{GATEWAY_ID}'
    print(f"\nGateway Created:")
    print(f"  Gateway ID: {GATEWAY_ID}")
    print(f"  Gateway URL: {GATEWAY_URL}")
    print(f"  RBAC Interceptor: Attached (REQUEST + RESPONSE)")
else:
    print("ERROR: Gateway creation failed")

## Step 6: Register Product Tools as Gateway Target

We register all 11 product tools (6 read + 5 admin) as a single Lambda target on the Gateway. The tool schemas tell the Gateway how to invoke each tool.

In [ ]:
from utils import create_lambda_gateway_target, get_product_tool_schemas

# Get all 11 tool schemas
TOOL_SCHEMAS = get_product_tool_schemas()
print(f"Registering {len(TOOL_SCHEMAS)} product tools as Gateway target:\n")
for schema in TOOL_SCHEMAS:
    print(f"  - {schema['name']}: {schema['description'][:60]}...")

# Create the Lambda target
print(f"\nCreating Gateway target...")
product_target = create_lambda_gateway_target(
    gateway_client,
    GATEWAY_ID,
    'ProductTools',
    PRODUCT_TOOLS_ARN,
    TOOL_SCHEMAS,
    'Product catalog tools - 6 read tools + 5 admin tools'
)

if product_target:
    print(f"Target created: ProductTools")
    print(f"Target ID: {product_target.get('targetId')}")
else:
    print("ERROR: Failed to create Gateway target")

## Step 7: Test Gateway Connection

Before deploying the agent container, let's verify the Gateway works by connecting directly with MCP and testing tool discovery for different roles.

In [ ]:
# Wait for Gateway to be ready
print("Waiting for Gateway resources to propagate...")
time.sleep(15)

# Get an M2M OAuth token for Gateway access
from utils import get_oauth_token

SCOPE_STRING = f"{COGNITO_RESOURCE_SERVER_ID}/gateway:read {COGNITO_RESOURCE_SERVER_ID}/gateway:write"

print("Getting OAuth token from Cognito...")
token_response = get_oauth_token(
    USER_POOL_ID, M2M_CLIENT_ID, M2M_CLIENT_SECRET, SCOPE_STRING, REGION
)

if 'access_token' in token_response:
    ACCESS_TOKEN = token_response['access_token']
    print(f"Access token obtained (expires in {token_response.get('expires_in', 'unknown')}s)")
else:
    print(f"Failed to get token: {token_response}")

In [ ]:
# Test MCP connection to Gateway - list tools
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client

print("Testing MCP connection to Gateway...\n")

def create_mcp_transport():
    return streamablehttp_client(
        GATEWAY_URL,
        headers={"Authorization": f"Bearer {ACCESS_TOKEN}"}
    )

mcp_client = MCPClient(create_mcp_transport)

with mcp_client:
    tools = mcp_client.list_tools_sync()
    print(f"Found {len(tools)} MCP tools via Gateway:\n")
    for tool in tools:
        print(f"  - {tool.tool_name}")

print("\nGateway connection successful!")

## Step 8: Test RBAC with Different User Tokens

Let's verify the RBAC interceptor works by getting tokens for both user roles and checking which tools each sees.

In [ ]:
from utils import get_user_token

# Get tokens for both test users
print("Getting JWT tokens for test users...\n")

customer_tokens = get_user_token(
    cognito_client, USER_POOL_ID, USER_CLIENT_ID,
    CUSTOMER_EMAIL, TEST_PASSWORD
)

admin_tokens = get_user_token(
    cognito_client, USER_POOL_ID, USER_CLIENT_ID,
    ADMIN_EMAIL, TEST_PASSWORD
)

# Decode and show claims
import base64

def decode_jwt_claims(token):
    parts = token.split('.')
    payload_b64 = parts[1]
    padding = 4 - len(payload_b64) % 4
    if padding != 4:
        payload_b64 += '=' * padding
    return json.loads(base64.urlsafe_b64decode(payload_b64))

if customer_tokens.get('id_token'):
    claims = decode_jwt_claims(customer_tokens['id_token'])
    print(f"Customer JWT claims:")
    print(f"  email: {claims.get('email')}")
    print(f"  cognito:groups: {claims.get('cognito:groups', [])}")

if admin_tokens.get('id_token'):
    claims = decode_jwt_claims(admin_tokens['id_token'])
    print(f"\nAdmin JWT claims:")
    print(f"  email: {claims.get('email')}")
    print(f"  cognito:groups: {claims.get('cognito:groups', [])}")

> **Note**: The RBAC interceptor reads the `Authorization` header's JWT token to determine the caller's role. The Gateway itself validates JWT signatures via the Cognito discovery URL. The interceptor only needs to decode the payload (no verification needed since the Gateway already validated it).

## Step 9: Build and Push Agent Container

AgentCore Runtime runs agents as containers. We'll build a Docker image for the Product Catalog Agent and push it to Amazon ECR.

The agent container:
- Uses `BedrockAgentCoreApp` as the entrypoint
- Connects to Gateway via SigV4-signed MCP client
- Extracts user role from JWT `cognito:groups` claim
- Filters tools by role (same pattern as Module 01)
- Runs on port 8080 with OpenTelemetry instrumentation

In [ ]:
import subprocess

# Create ECR repository
ecr_client = boto3.client('ecr', region_name=REGION)

print("Creating ECR repository...")
try:
    ecr_client.create_repository(
        repositoryName=ECR_REPO_NAME,
        imageScanningConfiguration={'scanOnPush': True},
        imageTagMutability='MUTABLE'
    )
    print(f"Created: {ECR_REPO_NAME}")
except ecr_client.exceptions.RepositoryAlreadyExistsException:
    print(f"Exists: {ECR_REPO_NAME}")

ECR_REGISTRY = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com"
CONTAINER_URI = f"{ECR_REGISTRY}/{ECR_REPO_NAME}:latest"
print(f"Container URI: {CONTAINER_URI}")

In [ ]:
# Login to ECR
print("Logging into ECR...")
login_cmd = f"aws ecr get-login-password --region {REGION} | docker login --username AWS --password-stdin {ECR_REGISTRY}"
result = subprocess.run(login_cmd, shell=True, capture_output=True, text=True)
if result.returncode == 0:
    print("ECR login successful")
else:
    print(f"ECR login failed: {result.stderr}")

In [ ]:
# Build Docker image (ARM64 required by AgentCore Runtime)
print("Building Docker image (ARM64)...")
print("This may take a few minutes on first build.\n")

build_cmd = f"docker build --platform linux/arm64 -t {ECR_REPO_NAME}:latest agents/"
result = subprocess.run(build_cmd, shell=True, capture_output=True, text=True)

if result.returncode == 0:
    print("Build successful!")
else:
    print(f"Build failed:\n{result.stderr}")

In [ ]:
# Tag and push to ECR
print("Pushing to ECR...")

tag_cmd = f"docker tag {ECR_REPO_NAME}:latest {CONTAINER_URI}"
subprocess.run(tag_cmd, shell=True, check=True)

push_cmd = f"docker push {CONTAINER_URI}"
result = subprocess.run(push_cmd, shell=True, capture_output=True, text=True)

if result.returncode == 0:
    print(f"Pushed successfully: {CONTAINER_URI}")
else:
    print(f"Push failed:\n{result.stderr}")

## Step 10: Create AgentCore Runtime

Deploy the containerized agent to AgentCore Runtime. The runtime provides:
- Managed container hosting with auto-scaling
- OpenTelemetry-based observability
- Secure endpoint for agent invocation

In [ ]:
from utils import create_agent_runtime_role

# Create Runtime IAM role with Gateway invocation permission
print("Creating AgentCore Runtime role...")
runtime_role_resp = create_agent_runtime_role(
    iam_client,
    RUNTIME_ROLE_NAME,
    gateway_arn=GATEWAY_ARN  # Permission to invoke Gateway tools
)
RUNTIME_ROLE_ARN = runtime_role_resp['Role']['Arn']
print(f"Runtime Role ARN: {RUNTIME_ROLE_ARN}")

In [ ]:
from utils import create_agent_runtime

agentcore_client = boto3.client('bedrock-agentcore-control', region_name=REGION)

# Agent environment variables
agent_env_vars = {
    'AGENT_REGION': REGION,
    'GATEWAY_URL': GATEWAY_URL,
    'MODEL_ID': 'global.anthropic.claude-haiku-4-5-20251001-v1:0'
}

print("Deploying Product Catalog Agent to AgentCore Runtime...")
print("This may take several minutes while the container starts up.\n")

runtime_response = create_agent_runtime(
    agentcore_client,
    runtime_name=RUNTIME_NAME,
    role_arn=RUNTIME_ROLE_ARN,
    container_uri=CONTAINER_URI,
    environment_vars=agent_env_vars,
    description='Product Catalog Agent with RBAC - single agent from Module 01'
)

if runtime_response:
    RUNTIME_ID = runtime_response['agentRuntimeId']
    RUNTIME_ARN = runtime_response['agentRuntimeArn']
    print(f"\nRuntime deployed successfully!")
    print(f"  Runtime ID: {RUNTIME_ID}")
    print(f"  Runtime ARN: {RUNTIME_ARN}")
    print(f"  Status: {runtime_response.get('status')}")
else:
    print("ERROR: Runtime deployment failed")

## Step 11: Test Production Agent - Customer Role

Now let's test the deployed agent with a **customer** JWT token. The customer should:
- Be able to use 6 read-only tools (search, details, inventory, recommendations, compare, return policy)
- NOT be able to use 5 admin tools (create, update, delete, update inventory, update pricing)

RBAC is enforced at two levels:
1. The agent filters tools based on JWT claims (application-level)
2. The Gateway interceptor blocks unauthorized tool calls (infrastructure-level)

In [ ]:
from utils import invoke_agent_runtime

agentcore_runtime_client = boto3.client('bedrock-agentcore', region_name=REGION)

# Test with customer token
customer_session_id = f"customer-{str(uuid.uuid4())}"

print("=" * 60)
print("Customer Test 1: Product Search (ALLOWED)")
print("=" * 60)

result = invoke_agent_runtime(
    agentcore_runtime_client,
    RUNTIME_ARN,
    customer_session_id,
    {
        'prompt': 'Search for wireless headphones under $100',
        'bearer_token': customer_tokens.get('id_token', ''),
        'access_token': customer_tokens.get('access_token', ''),
        'session_id': customer_session_id
    }
)

print(f"\nStatus: {result.get('status')}")
print(f"Role: {result.get('metadata', {}).get('role')}")
print(f"Tools available: {result.get('metadata', {}).get('tools_available')}")
print(f"Tools used: {result.get('metadata', {}).get('tools_used')}")
print(f"\nResponse:\n{result.get('response', result.get('error', 'No response'))}")

In [ ]:
# Customer Test 2: Attempt admin action (SHOULD BE REFUSED)
print("=" * 60)
print("Customer Test 2: Create Product (SHOULD BE REFUSED)")
print("=" * 60)

result = invoke_agent_runtime(
    agentcore_runtime_client,
    RUNTIME_ARN,
    customer_session_id,
    {
        'prompt': 'Create a new product called Super Speaker for $299.99 in Audio category',
        'bearer_token': customer_tokens.get('id_token', ''),
        'access_token': customer_tokens.get('access_token', ''),
        'session_id': customer_session_id
    }
)

print(f"\nStatus: {result.get('status')}")
print(f"Role: {result.get('metadata', {}).get('role')}")
print(f"Tools available: {result.get('metadata', {}).get('tools_available')}")
print(f"\nResponse:\n{result.get('response', result.get('error', 'No response'))}")
print("\n(Customer should be refused - create_product is admin only)")

In [ ]:
# Customer Test 2: Attempt admin action (SHOULD BE REFUSED)
print("=" * 60)
print("Customer Test 2: Create Product (SHOULD BE REFUSED)")
print("=" * 60)

result = invoke_agent_runtime(
    agentcore_runtime_client,
    RUNTIME_ARN,
    customer_session_id,
    {
        'prompt': 'Create a new product called Super Speaker for $299.99 in Audio category',
        'bearer_token': customer_tokens.get('id_token', ''),
        'session_id': customer_session_id
    }
)

print(f"\nStatus: {result.get('status')}")
print(f"Role: {result.get('metadata', {}).get('role')}")
print(f"Tools available: {result.get('metadata', {}).get('tools_available')}")
print(f"\nResponse:\n{result.get('response', result.get('error', 'No response'))}")
print("\n(Customer should be refused - create_product is admin only)")

# Admin Test 1: Search (admin has all customer tools too)

In [ ]:
admin_session_id = f"admin-{str(uuid.uuid4())}"

print("=" * 60)
print("Admin Test 1: Product Search (ALLOWED)")
print("=" * 60)

result = invoke_agent_runtime(
    agentcore_runtime_client,
    RUNTIME_ARN,
    admin_session_id,
    {
        'prompt': 'Search for headphone products',
        'bearer_token': admin_tokens.get('id_token', ''),
        'access_token': admin_tokens.get('access_token', ''),
        'session_id': admin_session_id
    }
)

print(f"\nStatus: {result.get('status')}")
print(f"Role: {result.get('metadata', {}).get('role')}")
print(f"Tools available: {result.get('metadata', {}).get('tools_available')}")
print(f"Tools used: {result.get('metadata', {}).get('tools_used')}")
print(f"\nResponse:\n{result.get('response', result.get('error', 'No response'))}")

In [ ]:
# Admin Test 2: Create a product (ALLOWED for admin)
print("=" * 60)
print("Admin Test 2: Create Product (ALLOWED)")
print("=" * 60)

result = invoke_agent_runtime(
    agentcore_runtime_client,
    RUNTIME_ARN,
    admin_session_id,
    {
        'prompt': (
            "Create a new product with ID PROD-201 called 'Premium Gaming Headset' "
            "in the Audio category for $129.99. Description: 'Professional gaming headset "
            "with 7.1 surround sound, detachable microphone, and RGB lighting.' "
            "Specifications: {\"bluetooth\": \"5.3\", \"weight\": \"320g\", \"driver_size\": \"50mm\"}. "
            "Set initial stock to 50 units."
        ),
        'bearer_token': admin_tokens.get('id_token', ''),
        'access_token': admin_tokens.get('access_token', ''),
        'session_id': admin_session_id
    }
)

print(f"\nStatus: {result.get('status')}")
print(f"Role: {result.get('metadata', {}).get('role')}")
print(f"Tools used: {result.get('metadata', {}).get('tools_used')}")
print(f"\nResponse:\n{result.get('response', result.get('error', 'No response'))}")

In [ ]:
# Admin Test 3: Set a sale price (ALLOWED for admin)
print("=" * 60)
print("Admin Test 3: Update Pricing (ALLOWED)")
print("=" * 60)

result = invoke_agent_runtime(
    agentcore_runtime_client,
    RUNTIME_ARN,
    admin_session_id,
    {
        'prompt': 'Set a sale price of $99.99 for PROD-201 (regular price stays $129.99) until 2025-06-30',
        'bearer_token': admin_tokens.get('id_token', ''),
        'access_token': admin_tokens.get('access_token', ''),
        'session_id': admin_session_id
    }
)

print(f"\nStatus: {result.get('status')}")
print(f"Tools used: {result.get('metadata', {}).get('tools_used')}")
print(f"\nResponse:\n{result.get('response', result.get('error', 'No response'))}")

In [ ]:
# Admin Test 4: Clean up - discontinue the test product
print("=" * 60)
print("Admin Test 4: Delete Product (ALLOWED - soft delete)")
print("=" * 60)

result = invoke_agent_runtime(
    agentcore_runtime_client,
    RUNTIME_ARN,
    admin_session_id,
    {
        'prompt': 'Discontinue product PROD-201',
        'bearer_token': admin_tokens.get('id_token', ''),
        'access_token': admin_tokens.get('access_token', ''),
        'session_id': admin_session_id
    }
)

print(f"\nStatus: {result.get('status')}")
print(f"Tools used: {result.get('metadata', {}).get('tools_used')}")
print(f"\nResponse:\n{result.get('response', result.get('error', 'No response'))}")

## Step 13: Save Configuration for Streamlit App

Save all deployment configuration so the Streamlit chat interface can connect to the production agent.

In [ ]:
from utils import save_config

# Build agent configuration
agent_config = {
    'runtime_arn': RUNTIME_ARN,
    'runtime_id': RUNTIME_ID,
    'gateway_id': GATEWAY_ID,
    'gateway_url': GATEWAY_URL,
    'region': REGION,
    'user_pool_id': USER_POOL_ID,
    'user_client_id': USER_CLIENT_ID,
    'customer_email': CUSTOMER_EMAIL,
    'admin_email': ADMIN_EMAIL,
    'test_password': TEST_PASSWORD,
    'model_id': 'global.anthropic.claude-haiku-4-5-20251001-v1:0'
}

save_config(agent_config, 'streamlit_app/agent_config.json')

print("\nConfiguration saved for Streamlit app.")
print("\nTo launch the chat interface:")
print("  cd streamlit_app")
print("  pip install streamlit")
print("  streamlit run app.py")
print("\nThe app will be available at http://localhost:8501")

## Step 14: Deployment Summary

Let's review everything that was deployed.

In [ ]:
print("=" * 60)
print("DEPLOYMENT COMPLETE!")
print("=" * 60)

print(f"\nInfrastructure:")
print(f"  AgentCore Runtime: {RUNTIME_ARN}")
print(f"  AgentCore Gateway: {GATEWAY_URL}")
print(f"  RBAC Interceptor:  {INTERCEPTOR_ARN}")
print(f"  Cognito Pool:      {USER_POOL_ID}")

print(f"\nMCP Tools (11 total):")
print(f"  Read tools (6):  search_products, get_product_details, check_inventory,")
print(f"                   get_product_recommendations, compare_products, get_return_policy")
print(f"  Admin tools (5): create_product, update_product, delete_product,")
print(f"                   update_inventory, update_pricing")

print(f"\nRBAC:")
print(f"  Customer ({CUSTOMER_EMAIL}): 6 read-only tools")
print(f"  Admin ({ADMIN_EMAIL}):    11 tools (6 read + 5 admin)")
print(f"  Enforcement: Agent-side filtering + Gateway interceptor")

print(f"\nStreamlit App: streamlit_app/app.py")
print("=" * 60)

In [ ]:
# Save deployment info for subsequent modules
deployment_info = {
    'runtime_arn': RUNTIME_ARN,
    'runtime_id': RUNTIME_ID,
    'gateway_id': GATEWAY_ID,
    'gateway_url': GATEWAY_URL,
    'user_pool_id': USER_POOL_ID,
    'region': REGION
}

%store deployment_info
%store REGION
print("Deployment information saved for subsequent modules")

---

## Cleanup (Optional)

Run the cells below to delete all resources created in this module. **Only run this when you're done with the workshop.**

In [ ]:
# UNCOMMENT AND RUN TO CLEAN UP ALL RESOURCES

# from utils import delete_agent_runtime, delete_gateway
#
# print("Cleaning up Module 03 resources...\n")
#
# # 1. Delete AgentCore Runtime
# print("Deleting AgentCore Runtime...")
# delete_agent_runtime(agentcore_client, RUNTIME_ID)
#
# # 2. Delete AgentCore Gateway (and targets)
# print("\nDeleting AgentCore Gateway...")
# delete_gateway(gateway_client, GATEWAY_ID)
#
# # 3. Delete Lambda functions
# print("\nDeleting Lambda functions...")
# lambda_client = boto3.client('lambda', region_name=REGION)
# for func_name in [f'{WORKSHOP_PREFIX}-product-tools', f'{WORKSHOP_PREFIX}-rbac-interceptor']:
#     try:
#         lambda_client.delete_function(FunctionName=func_name)
#         print(f"  Deleted: {func_name}")
#     except Exception as e:
#         print(f"  Error deleting {func_name}: {e}")
#
# # 4. Delete Cognito User Pool
# print("\nDeleting Cognito User Pool...")
# try:
#     cognito_client.delete_user_pool_domain(Domain=DOMAIN_PREFIX, UserPoolId=USER_POOL_ID)
#     cognito_client.delete_user_pool(UserPoolId=USER_POOL_ID)
#     print(f"  Deleted pool: {USER_POOL_ID}")
# except Exception as e:
#     print(f"  Error: {e}")
#
# # 5. Delete ECR repository
# print("\nDeleting ECR repository...")
# try:
#     ecr_client.delete_repository(repositoryName=ECR_REPO_NAME, force=True)
#     print(f"  Deleted: {ECR_REPO_NAME}")
# except Exception as e:
#     print(f"  Error: {e}")
#
# # 6. Delete IAM roles
# print("\nDeleting IAM roles...")
# for role_name in [LAMBDA_ROLE_NAME, GATEWAY_ROLE_NAME, RUNTIME_ROLE_NAME]:
#     try:
#         # Detach managed policies
#         policies = iam_client.list_attached_role_policies(RoleName=role_name)
#         for policy in policies['AttachedPolicies']:
#             iam_client.detach_role_policy(RoleName=role_name, PolicyArn=policy['PolicyArn'])
#         # Delete inline policies
#         inline = iam_client.list_role_policies(RoleName=role_name)
#         for policy_name in inline['PolicyNames']:
#             iam_client.delete_role_policy(RoleName=role_name, PolicyName=policy_name)
#         iam_client.delete_role(RoleName=role_name)
#         print(f"  Deleted: {role_name}")
#     except Exception as e:
#         print(f"  Error deleting {role_name}: {e}")
#
# print("\nCleanup complete!")

---

## Module 3 Summary

You deployed the Product Catalog Agent from Module 01 to production using Amazon Bedrock AgentCore.

### What Was Created

| Component | Description |
|-----------|-------------|
| **Product Tools Lambda** | 11 MCP tools (6 read + 5 admin) as a single Lambda |
| **RBAC Interceptor Lambda** | Gateway interceptor for infrastructure-level access control |
| **Cognito User Pool** | User management with `customer` and `admin` groups |
| **AgentCore Gateway** | MCP endpoint with JWT auth and RBAC interceptor |
| **ECR Repository** | Container image for the Product Catalog Agent |
| **AgentCore Runtime** | Managed hosting for the containerized agent |

### Defense-in-Depth RBAC

| Layer | Mechanism | What It Does |
|-------|-----------|-------------|
| **Application** | Agent tool filtering | Removes admin tools from customer agent (same as Module 01) |
| **Infrastructure** | Gateway interceptor | Blocks admin tool calls and hides admin tools in discovery |

### Key Files

| File | Description |
|------|-------------|
| `lambda_tools/product_tools_lambda.py` | 11 product tools for Gateway Lambda target |
| `lambda_tools/rbac_interceptor_lambda.py` | RBAC interceptor (REQUEST + RESPONSE) |
| `agents/product_catalog_agent.py` | Production agent with JWT role extraction |
| `agents/Dockerfile` | ARM64 container for AgentCore Runtime |
| `utils.py` | Deployment utility functions |
| `streamlit_app/app.py` | Chat UI with Cognito login |

### Prototype to Production Mapping

| Module 01 Pattern | Module 03 Implementation |
|-------------------|-------------------------|
| `UserSession(role='customer')` | JWT with `cognito:groups: ['customer']` |
| `UserSession(role='admin')` | JWT with `cognito:groups: ['admin']` |
| `MCPClient(stdio_client(...))` | `MCPClient(streamablehttp_client(...))` with SigV4 |
| `mcp_server.py` (local process) | `product_tools_lambda.py` (Lambda behind Gateway) |
| Agent-side tool filtering only | Agent filtering + Gateway interceptor |

### Next Steps

1. Launch the Streamlit app to test the chat interface with different user roles
2. Monitor agent performance in CloudWatch logs
3. Explore observability with OpenTelemetry traces